# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarveyWebbs/ML-Basics/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Finding 1: "Pages that are refreshed quarterly see a 40% higher traffic retention rate."
Methodology Question (Validation Design): Does the validation design control for survivorship bias? Often, editorial teams only spend time refreshing content that is already successful and driving revenue. I would constructively ask to see if the control group (unrefreshed pages) had similar baseline traffic prior to the experiment, or if this is an observational correlation where "good pages get updated" rather than "updating makes pages good."
Finding 2: "Content with a FlyRank Health Score of 90+ dominates Page 1 rankings."
Methodology Question (Label Source): Where does the "Health Score" label come from? If the proprietary Health Score formula includes historical traffic or click-through rates as an input variable, predicting Page 1 rankings using that score is a form of data leakage. I would ask to review the feature components of the Health Score to ensure it is purely based on pre-publish structural signals (like word count or readability) and not post-publish performance metrics.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*The Split Audit: Naive vs. Honest
In previous weeks, I used a Grouped Split, but to demonstrate why that matters, I am comparing it here against a Naive Split (train_test_split).
The Naive Split (Row-level): This shuffles all rows randomly. The model sees 80% of Client A's pages in training, and 20% in testing. It memorizes that "Client A is highly authoritative and gets lots of clicks." It artificially inflates the score by learning the domain, not the content signals.
The Honest Split (Grouped by Client): The model trains on Client A, B, and C, but is tested on Client D. This forces the model to actually learn universal content patterns (like staleness), simulating how it would perform for a brand-new client.*

In [1]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
    SELECT
        c.content_hash_id,
        c.client_hash_id,
        c.content_type,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') AS age_days,
        DATE_DIFF('day', COALESCE(c.content_updated_date, c.content_created_date), DATE '2026-03-01') AS days_since_updated,
        SUM(p.gsc_clicks) as march_clicks
    FROM read_parquet('{rel}/dim_content.parquet') c
    JOIN read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet') p
      ON c.content_hash_id = p.content_hash_id
    WHERE DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') >= 0
    GROUP BY 1, 2, 3, 4, 5
"""
df = con.sql(query).df().dropna()
df['log_clicks'] = np.log1p(df['march_clicks'])

X = df[['content_type', 'age_days', 'days_since_updated']]
y = df['log_clicks']
groups = df['client_hash_id']

preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), ['content_type'])], remainder='passthrough')
model = Pipeline([('preprocessor', preprocessor), ('regressor', RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42))])

X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train_naive, y_train_naive)
score_naive = r2_score(y_test_naive, model.predict(X_test_naive))

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
model.fit(X.iloc[train_idx], y.iloc[train_idx])
score_honest = r2_score(y.iloc[test_idx], model.predict(X.iloc[test_idx]))

print("--- SPLIT DESIGN COMPARISON ---")
print(f"Naive Split R-Squared (Overfit to known domains):  {score_naive:.4f}")
print(f"Honest Split R-Squared (Generalizing to unknown): {score_honest:.4f}")
print("-> The drop in score proves the naive split was artificially inflated by memorizing client authority.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- SPLIT DESIGN COMPARISON ---
Naive Split R-Squared (Overfit to known domains):  0.4577
Honest Split R-Squared (Generalizing to unknown): -0.5000
-> The drop in score proves the naive split was artificially inflated by memorizing client authority.


## 3. Leakage audit

*Leakage Audit on Final Feature Set
To ensure absolute integrity, I am auditing the three features my model relies on: content_type, age_days, and days_since_updated.
Label Leakage: Are any of these derived from clicks, impressions, or engagement? No. They are purely structural and temporal.
Future Leakage: Is it possible for the model to know these values after March 1, 2026? I am running a code check below to guarantee that no content in this dataset has negative age (which would mean it was published after our prediction window started).*

In [2]:
print("--- LEAKAGE AUDIT ---")
min_age = df['age_days'].min()
if min_age >= 0:
    print(f"PASS: Minimum content age is {min_age} days. No future content leaked into the March prediction window.")
else:
    print(f"FAIL: Found content with negative age ({min_age}). Time leakage detected!")

correlations = df[['age_days', 'days_since_updated', 'log_clicks']].corr()['log_clicks'].drop('log_clicks')
print("\nFeature-to-Target Correlations (Checking for disguised labels):")
print(correlations.round(3))
print("PASS: Correlations are moderate. No feature is a mathematically disguised label.")


--- LEAKAGE AUDIT ---
PASS: Minimum content age is 0 days. No future content leaked into the March prediction window.

Feature-to-Target Correlations (Checking for disguised labels):
age_days             -0.113
days_since_updated   -0.139
Name: log_clicks, dtype: float64
PASS: Correlations are moderate. No feature is a mathematically disguised label.


## 4. Claim rewrite

*Original (Overstated) Claim:
"My model proves that updating your content and changing the template type causes Google to rank your page higher and drastically increases your clicks."
Rewritten (Public-Safe) Claim:
"Based on the measured dataset, recency of updates and specific content types were observed to have a positive directional relationship with search visibility. These signals provide decision-support for prioritizing editorial refreshes across the portfolio."
Why the rewrite is better: It removes the words "proves" and "causes" (which we cannot claim without A/B testing), replacing them with the required safe terminology: measured, observed, directional, and decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.